In [ ]:
!pip install -q -U transformers datasets accelerate evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU available: True
Tesla T4


In [ ]:
from google.colab import files
uploaded = files.upload()  # select databricks-dolly-15k.jsonl from your computer

Saving databricks-dolly-15k.jsonl to databricks-dolly-15k.jsonl


In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

records = []
with open("databricks-dolly-15k.jsonl", "r") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(df.shape)
print(df["category"].value_counts())
df.head()

(15015, 4)
category
open_qa                   3611
general_qa                2191
classification            2136
closed_qa                 1823
brainstorming             1768
information_extraction    1512
summarization             1263
creative_writing           711
Name: count, dtype: int64


,instruction,context,response,category
0,When did Virgin Australia start operating?,"Virgin Australia, the trading name of Virgin A...",Virgin Australia commenced services on 31 Augu...,closed_qa
1,Which is a species of fish? Tope or Rope,,Tope,classification
2,Why can camels survive for long without water?,,Camels use the fat in their humps to keep them...,open_qa
3,"Alice's parents have three daughters: Amy, Jes...",,The name of the third daughter is Alice,open_qa
4,When was Tomoaki Komorida born?,Komorida was born in Kumamoto Prefecture on Ju...,"Tomoaki Komorida was born on July 10,1981.",closed_qa


In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant. Answer the user's instruction accurately and concisely."

def build_user_message(row):
    if row["context"].strip():
        return f"{row['instruction']}\n\nContext:\n{row['context']}"
    return row["instruction"]

df["user_message"] = df.apply(build_user_message, axis=1)
df = df[["user_message", "response"]].rename(columns={"response": "assistant_message"})

# Drop any empty rows just in case
df = df[(df["user_message"].str.strip() != "") & (df["assistant_message"].str.strip() != "")]
print(df.shape)

(15015, 2)


In [ ]:
from sklearn.model_selection import train_test_split

# Use a subset for faster training in Colab; increase these if you have time/compute
N_TRAIN = 3000
N_VAL = 500
N_TEST = 500
SEED = 42

train_df, temp_df = train_test_split(df, test_size=(N_VAL + N_TEST) / len(df), random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=N_TEST / (N_VAL + N_TEST), random_state=SEED)

train_df = train_df.sample(n=min(N_TRAIN, len(train_df)), random_state=SEED).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 3000 | Val: 500 | Test: 501


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()

for param in model.parameters():
    param.requires_grad = True

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Total parameters: 361,821,120


In [ ]:
MAX_LENGTH = 256

def tokenize_example(user_message, assistant_message):
    messages_prompt_only = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages_prompt_only, tokenize=False, add_generation_prompt=True
    )

    messages_full = messages_prompt_only + [{"role": "assistant", "content": assistant_message}]
    full_text = tokenizer.apply_chat_template(
        messages_full, tokenize=False, add_generation_prompt=False
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(
        full_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH
    )["input_ids"]

    labels = list(full_ids)
    prompt_len = min(len(prompt_ids), len(full_ids))
    for i in range(prompt_len):
        labels[i] = -100  # mask prompt tokens from the loss

    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


def build_dataset(dataframe):
    rows = [
        tokenize_example(r["user_message"], r["assistant_message"])
        for _, r in dataframe.iterrows()
    ]
    return Dataset.from_list(rows)


train_ds = build_dataset(train_df)
val_ds = build_dataset(val_df)
test_ds = build_dataset(test_df)

print(train_ds)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

In [ ]:
def compute_metrics(eval_pred):
    # Trainer already computes eval_loss; we add perplexity derived from it
    # (this function only needs to return an empty dict here — perplexity is
    # added manually after evaluate() is called, see Cell 13)
    return {}

In [ ]:
training_args = TrainingArguments(
    output_dir="results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    prediction_loss_only=True,
    eval_accumulation_steps=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.174040,nan
2,2.155139,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,2.174040,nan
2,2.155139,nan
3,2.233252,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=564, training_loss=2.2557294148925346, metrics={'train_runtime': 1607.9143, 'train_samples_per_second': 5.597, 'train_steps_per_second': 0.351, 'total_flos': 3338567677843200.0, 'train_loss': 2.2557294148925346, 'epoch': 3.0})

In [ ]:
import math

val_metrics = trainer.evaluate(eval_dataset=val_ds)
val_metrics["perplexity"] = math.exp(val_metrics["eval_loss"])
print("Validation metrics:", val_metrics)

test_metrics = trainer.evaluate(eval_dataset=test_ds)
test_metrics["perplexity"] = math.exp(test_metrics["eval_loss"])
print("Test metrics:", test_metrics)

Training Loss,Validation Loss,Epoch
2.233252,nan,3


Validation metrics: {'eval_loss': nan, 'perplexity': nan}


Training Loss,Validation Loss,Epoch
2.233252,nan,3


Test metrics: {'eval_loss': nan, 'perplexity': nan}


In [ ]:
import evaluate

rouge = evaluate.load("rouge")

def generate_response(user_message, max_new_tokens=150):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return generated.strip()


# Use a subset of the test set for speed (generation is slower than a forward pass)
N_GEN_EVAL = 100
eval_subset = test_df.sample(n=min(N_GEN_EVAL, len(test_df)), random_state=SEED)

predictions, references = [], []
for _, row in eval_subset.iterrows():
    pred = generate_response(row["user_message"])
    predictions.append(pred)
    references.append(row["assistant_message"])

rouge_scores = rouge.compute(predictions=predictions, references=references)
print("ROUGE scores on test subset:", rouge_scores)

ROUGE scores on test subset: {'rouge1': np.float64(0.3302641674009833), 'rouge2': np.float64(0.1640135352309987), 'rougeL': np.float64(0.2668601018643464), 'rougeLsum': np.float64(0.2786293026613423)}


In [ ]:
SAVE_DIR = "smollm2_dolly_finetuned"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to ./{SAVE_DIR}")

# Optional: zip and download, or copy to Google Drive
!zip -r smollm2_dolly_finetuned.zip smollm2_dolly_finetuned
from google.colab import files
files.download("smollm2_dolly_finetuned.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./smollm2_dolly_finetuned
  adding: smollm2_dolly_finetuned/ (stored 0%)
  adding: smollm2_dolly_finetuned/chat_template.jinja (deflated 42%)
  adding: smollm2_dolly_finetuned/model.safetensors (deflated 7%)
  adding: smollm2_dolly_finetuned/tokenizer_config.json (deflated 49%)
  adding: smollm2_dolly_finetuned/generation_config.json (deflated 27%)
  adding: smollm2_dolly_finetuned/config.json (deflated 52%)
  adding: smollm2_dolly_finetuned/tokenizer.json (deflated 82%)
  adding: smollm2_dolly_finetuned/training_args.bin (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

loaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
loaded_model = AutoModelForCausalLM.from_pretrained(SAVE_DIR).to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

def ask(question, context=""):
    user_msg = f"{question}\n\nContext:\n{context}" if context.strip() else question
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    prompt = loaded_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = loaded_tokenizer(prompt, return_tensors="pt").to(loaded_model.device)
    with torch.no_grad():
        output_ids = loaded_model.generate(
            **inputs, max_new_tokens=200, do_sample=False,
            pad_token_id=loaded_tokenizer.pad_token_id,
        )
    return loaded_tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

print(ask("What is the capital of France?"))
print(ask("Summarize this text.", context="The Eiffel Tower is a wrought-iron lattice tower in Paris, France..."))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The capital of France is Paris.
The Eiffel Tower is a wrought-iron lattice tower in Paris, France.
